In [1]:
"""
Financial GAN with WGAN-GP and Enhanced Architecture
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from scipy.stats import ks_2samp

# Enable debug mode to print shapes
DEBUG = True

def debug_print(message, tensor=None):
    """Helper function for debug printing"""
    if DEBUG:
        if tensor is not None:
            print(f"{message}: {tensor.shape}")
        else:
            print(message)

class FinancialDataset(Dataset):
    def __init__(self, data_dict, sequence_length=20):
        """
        Parameters:
        data_dict (dict): Dictionary of DataFrames with OHLCV data
        sequence_length (int): Length of sequences to generate
        """
        self.sequence_length = sequence_length
        self.data = []
        self.tickers = list(data_dict.keys())
        self.stats = {}
        
        for ticker in self.tickers:
            df = data_dict[ticker]
            normalized_data, stats = self._normalize_data(df)
            self.data.append(normalized_data)
            self.stats[ticker] = stats
            
        self.data = torch.stack(self.data)  # [n_stocks, n_days, n_features]
        
        if DEBUG:
            print(f"Dataset initialized with shape: {self.data.shape}")
            print(f"Number of stocks: {len(self.tickers)}")
    
    def _normalize_data(self, df):
        """Normalize OHLCV data"""
        data = df[['Open', 'High', 'Low', 'Close', 'Volume']].values
        data[:, -1] = np.log(data[:, -1] + 1)  # Log transform volume
        
        stats = {
            'min': data.min(axis=0),
            'max': data.max(axis=0),
            'mean': data.mean(axis=0),
            'std': data.std(axis=0)
        }
        
        normalized = (data - stats['min']) / (stats['max'] - stats['min'])
        return torch.FloatTensor(normalized), stats
    
    def denormalize(self, normalized_data, ticker):
        """Denormalize data back to original scale"""
        stats = self.stats[ticker]
        denorm_data = normalized_data * (stats['max'] - stats['min']) + stats['min']
        # Reverse log transform for volume
        denorm_data[..., -1] = np.exp(denorm_data[..., -1]) - 1
        return denorm_data
    
    def __len__(self):
        return len(self.data[0]) - self.sequence_length
    
    def __getitem__(self, idx):
        sequence = self.data[:, idx:idx+self.sequence_length, :]  # [n_stocks, seq_len, n_features]
        return sequence

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class FinancialLoss(nn.Module):
    def __init__(self, lambda_gp=10.0, lambda_fin=0.1):
        super().__init__()
        self.lambda_gp = lambda_gp
        self.lambda_fin = lambda_fin
        self.eps = 0.00001  # Small epsilon for numerical stability
        
    def gradient_penalty(self, discriminator, real_data, fake_data, device):
        batch_size = real_data.size(0)
        
        # Random weight term for interpolation
        alpha = torch.rand(batch_size, 1, 1, 1).to(device)
        alpha = alpha.expand_as(real_data)
        
        # Get random interpolation between real and fake data
        interpolated = alpha * real_data + (1 - alpha) * fake_data
        interpolated = interpolated.requires_grad_(True)
        
        # Calculate probability of interpolated examples
        d_interpolated = discriminator(interpolated)
        
        # Calculate gradients of probabilities with respect to examples
        gradients = torch.autograd.grad(
            outputs=d_interpolated,
            inputs=interpolated,
            grad_outputs=torch.ones_like(d_interpolated).to(device),
            create_graph=True,
            retain_graph=True,
        )[0]
        
        # Calculate gradient penalty
        gradients = gradients.view(batch_size, -1)
        gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
        
        return gradient_penalty
    
    def price_consistency(self, data):
        """Ensure OHLC price consistency"""
        open_price = data[..., 0]
        high_price = data[..., 1]
        low_price = data[..., 2]
        close_price = data[..., 3]
        
        high_low = F.relu(low_price - high_price).mean()
        open_range = F.relu(open_price - high_price).mean() + F.relu(low_price - open_price).mean()
        close_range = F.relu(close_price - high_price).mean() + F.relu(low_price - close_price).mean()
        
        return (high_low + open_range + close_range).clamp(min=self.eps)
    
    def temporal_coherence(self, data):
        """Ensure smooth price transitions"""
        returns = torch.diff(data[..., 3], dim=-1)  # Using close price
        acceleration = torch.diff(returns, dim=-1)
        return torch.abs(acceleration).mean().clamp(min=self.eps)
    
    def volume_price_correlation(self, data):
        """Encourage volume-volatility relationship"""
        returns = torch.diff(data[..., 3], dim=-1)
        volumes = data[..., 4][..., 1:]
        abs_returns = torch.abs(returns)
        
        # Normalize with stable computations
        abs_returns = (abs_returns - abs_returns.mean()) / (abs_returns.std() + self.eps)
        volumes = (volumes - volumes.mean()) / (volumes.std() + self.eps)
        
        return (-torch.mean(abs_returns * volumes)).clamp(min=self.eps)

In [2]:
class Generator(nn.Module):
    def __init__(self, n_stocks, n_features, noise_dim=100, d_model=256, nhead=8, num_layers=6):
        super().__init__()
        self.d_model = d_model
        self.noise_dim = noise_dim
        self.n_stocks = n_stocks
        
        # Initial noise projection
        self.noise_proj = nn.Sequential(
            nn.Linear(noise_dim, d_model),
            nn.LayerNorm(d_model),
            nn.LeakyReLU(0.2)
        )
        
        # Time embedding
        self.time_embedding = nn.Parameter(torch.randn(1, 1, d_model))
        
        # Stock embedding
        self.stock_embedding = nn.Parameter(torch.randn(1, n_stocks, 1, d_model))
        
        # Transformer encoder
        encoder_layers = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=1024,
            dropout=0.1,
            activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layers, num_layers)
        
        # Price generator
        self.price_generator = nn.Sequential(
            nn.Linear(d_model, 512),
            nn.LayerNorm(512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 4),  # OHLC
            nn.Sigmoid()
        )
        
        # Volume generator
        self.volume_generator = nn.Sequential(
            nn.Linear(d_model + 4, 256),
            nn.LayerNorm(256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )
    
    def forward(self, noise, seq_len):
        batch_size = noise.size(0)
        device = noise.device
        
        # Project noise
        x = self.noise_proj(noise)  # [batch_size, d_model]
        debug_print("After noise projection", x)
        
        # Add time dimension
        x = x.unsqueeze(1).repeat(1, seq_len, 1)  # [batch_size, seq_len, d_model]
        debug_print("After adding time dimension", x)
        
        # Add time embedding
        position_ids = torch.arange(seq_len, device=device).float().unsqueeze(0)
        time_embed = self.time_embedding * position_ids.unsqueeze(-1)
        x = x + time_embed
        debug_print("After time embedding", x)
        
        # Add stock dimension and embedding
        x = x.unsqueeze(1).expand(-1, self.n_stocks, -1, -1)  # [batch_size, n_stocks, seq_len, d_model]
        x = x + self.stock_embedding
        debug_print("After stock embedding", x)
        
        # Reshape for transformer
        x = x.permute(2, 0, 1, 3)  # [seq_len, batch_size, n_stocks, d_model]
        orig_shape = x.shape
        x = x.reshape(seq_len, -1, self.d_model)
        debug_print("Before transformer", x)
        
        # Transform
        x = self.transformer(x)
        debug_print("After transformer", x)
        
        # Reshape back
        x = x.reshape(*orig_shape)  # [seq_len, batch_size, n_stocks, d_model]
        x = x.permute(1, 2, 0, 3)  # [batch_size, n_stocks, seq_len, d_model]
        debug_print("After reshaping back", x)
        
        # Generate prices
        prices = self.price_generator(x)  # [batch_size, n_stocks, seq_len, 4]
        debug_print("Generated prices", prices)
        
        # Generate volume using price information
        volume_input = torch.cat([x, prices], dim=-1)
        volume = self.volume_generator(volume_input)  # [batch_size, n_stocks, seq_len, 1]
        debug_print("Generated volume", volume)
        
        # Combine prices and volume
        output = torch.cat([prices, volume], dim=-1)  # [batch_size, n_stocks, seq_len, 5]
        debug_print("Final output", output)
        
        return output

class Discriminator(nn.Module):
    def __init__(self, n_stocks, n_features, d_model=256, nhead=8, num_layers=6):
        super().__init__()
        self.d_model = d_model
        self.n_stocks = n_stocks
        
        # Feature extraction
        self.feature_extractor = nn.Sequential(
            nn.Linear(n_features, d_model),
            nn.LayerNorm(d_model),
            nn.LeakyReLU(0.2)
        )
        
        # Time embedding
        self.time_embedding = nn.Parameter(torch.randn(1, 1, 1, d_model))
        
        # Stock embedding
        self.stock_embedding = nn.Parameter(torch.randn(1, n_stocks, 1, d_model))
        
        # Transformer encoder
        encoder_layers = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=1024,
            dropout=0.1,
            activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layers, num_layers)
        
        # Output projection
        self.output_proj = nn.Sequential(
            nn.Linear(d_model, 512),
            nn.LayerNorm(512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 1)
        )
    
    def forward(self, x):
        # Input shape: [batch_size, n_stocks, seq_len, features]
        batch_size, n_stocks, seq_len, _ = x.shape
        device = x.device
        debug_print("Discriminator input", x)
        
        # Extract features
        x = self.feature_extractor(x)  # [batch_size, n_stocks, seq_len, d_model]
        debug_print("After feature extraction", x)
        
        # Add time embedding
        position_ids = torch.arange(seq_len, device=device).float().reshape(1, 1, -1, 1)
        time_embed = self.time_embedding * position_ids
        x = x + time_embed
        debug_print("After time embedding", x)
        
        # Add stock embedding
        x = x + self.stock_embedding
        debug_print("After stock embedding", x)
        
        # Reshape for transformer
        x = x.permute(2, 0, 1, 3)  # [seq_len, batch_size, n_stocks, d_model]
        x = x.reshape(seq_len, batch_size * n_stocks, self.d_model)
        debug_print("Before transformer", x)
        
        # Transform
        x = self.transformer(x)
        debug_print("After transformer", x)
        
        # Global average pooling
        x = x.mean(0)  # [batch_size*n_stocks, d_model]
        debug_print("After pooling", x)
        
        # Output projection
        out = self.output_proj(x)
        debug_print("Final output", out)
        
        return out

In [3]:
class FinancialGAN:
    def __init__(self, data_dict, sequence_length=20, batch_size=32, noise_dim=100):
        self.data_dict = data_dict
        self.sequence_length = sequence_length
        self.batch_size = batch_size
        self.noise_dim = noise_dim
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        # Initialize dataset
        self.dataset = FinancialDataset(data_dict, sequence_length)
        self.dataloader = DataLoader(
            self.dataset, 
            batch_size=batch_size, 
            shuffle=True,
            drop_last=True  # Ensure consistent batch sizes
        )
        
        # Get dimensions
        n_stocks = len(data_dict)
        n_features = 5  # OHLCV
        
        debug_print(f"Initializing GAN with {n_stocks} stocks")
        
        # Initialize models
        self.generator = Generator(n_stocks, n_features, noise_dim).to(self.device)
        self.discriminator = Discriminator(n_stocks, n_features).to(self.device)
        
        # Initialize optimizers
        self.g_optimizer = torch.optim.AdamW(
            self.generator.parameters(),
            lr=0.0001,
            betas=(0.5, 0.999),
            weight_decay=0.01
        )
        self.d_optimizer = torch.optim.AdamW(
            self.discriminator.parameters(),
            lr=0.0003,
            betas=(0.5, 0.999),
            weight_decay=0.01
        )
        
        # Initialize loss
        self.financial_loss = FinancialLoss()
    
    def train_step(self, real_data):
        batch_size = real_data.size(0)
        real_data = real_data.to(self.device)
        
        debug_print("Initial real_data", real_data)
        
        # Train Discriminator
        for _ in range(5):  # Multiple D updates per G update
            self.d_optimizer.zero_grad()
            
            # Generate fake data
            noise = torch.randn(batch_size, self.noise_dim).to(self.device)
            fake_data = self.generator(noise, self.sequence_length)
            
            debug_print("Generated fake_data", fake_data)
            
            # Get discriminator outputs
            d_real = self.discriminator(real_data)
            d_fake = self.discriminator(fake_data.detach())
            
            # Calculate discriminator loss
            d_loss = torch.mean(d_fake) - torch.mean(d_real)
            gradient_penalty = self.financial_loss.gradient_penalty(
                self.discriminator, real_data, fake_data, self.device
            )
            d_loss = d_loss + self.financial_loss.lambda_gp * gradient_penalty
            
            # Optimize discriminator
            d_loss.backward(retain_graph=True)
            torch.nn.utils.clip_grad_norm_(self.discriminator.parameters(), 1.0)
            self.d_optimizer.step()
        
        # Train Generator
        self.g_optimizer.zero_grad()
        
        fake_data = self.generator(noise, self.sequence_length)
        d_fake = self.discriminator(fake_data)
        
        # Calculate generator losses
        g_loss_gan = -torch.mean(d_fake)
        
        # Financial constraints
        price_consistency = self.financial_loss.price_consistency(fake_data)
        temporal_coherence = self.financial_loss.temporal_coherence(fake_data)
        volume_correlation = self.financial_loss.volume_price_correlation(fake_data)
        
        g_loss = (g_loss_gan + 
                 self.financial_loss.lambda_fin * (
                     price_consistency + 
                     temporal_coherence + 
                     volume_correlation
                 ))
        
        # Optimize generator
        g_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.generator.parameters(), 1.0)
        self.g_optimizer.step()
        
        return d_loss.item(), g_loss.item()
    
    def train(self, epochs=100):
        history = {
            'd_loss': [],
            'g_loss': []
        }
        
        for epoch in tqdm(range(epochs)):
            epoch_d_loss = []
            epoch_g_loss = []
            
            for batch_idx, real_data in enumerate(self.dataloader):
                try:
                    d_loss, g_loss = self.train_step(real_data)
                    
                    if np.isnan(d_loss) or np.isnan(g_loss):
                        print(f"NaN detected at epoch {epoch}, batch {batch_idx}")
                        continue
                        
                    epoch_d_loss.append(d_loss)
                    epoch_g_loss.append(g_loss)
                    
                except RuntimeError as e:
                    print(f"Error in batch {batch_idx}: {str(e)}")
                    continue
            
            # Calculate epoch metrics
            avg_d_loss = np.mean(epoch_d_loss) if epoch_d_loss else float('nan')
            avg_g_loss = np.mean(epoch_g_loss) if epoch_g_loss else float('nan')
            
            history['d_loss'].append(avg_d_loss)
            history['g_loss'].append(avg_g_loss)
            
            if epoch % 10 == 0:
                print(f'Epoch [{epoch}/{epochs}] '
                      f'D_loss: {avg_d_loss:.4f} '
                      f'G_loss: {avg_g_loss:.4f}')
        
        return history
    
    def generate_synthetic_data(self, n_samples=100):
        self.generator.eval()
        with torch.no_grad():
            noise = torch.randn(n_samples, self.noise_dim).to(self.device)
            synthetic_data = self.generator(noise, self.sequence_length)
            synthetic_data = synthetic_data.cpu().numpy()
        
        synthetic_dict = {}
        for stock_idx, stock_name in enumerate(self.data_dict.keys()):
            denorm_data = self.dataset.denormalize(
                synthetic_data[:, stock_idx], 
                stock_name
            )
            
            synthetic_df = pd.DataFrame(
                denorm_data,
                columns=['Open', 'High', 'Low', 'Close', 'Volume']
            )
            synthetic_dict[stock_name] = synthetic_df
        
        return synthetic_dict

def plot_results(real_dict, synthetic_dict):
    """Plot comparison between real and synthetic data"""
    n_stocks = len(real_dict)
    fig, axes = plt.subplots(n_stocks, 3, figsize=(20, 6*n_stocks))
    
    for idx, (ticker, real_df) in enumerate(real_dict.items()):
        synthetic_df = synthetic_dict[ticker]
        
        # Price plot
        ax = axes[idx, 0]
        ax.plot(real_df['Close'].values[:100], label='Real', alpha=0.7)
        ax.plot(synthetic_df['Close'].values[:100], label='Synthetic', alpha=0.7)
        ax.set_title(f'{ticker} - Price Comparison')
        ax.legend()
        
        # Return distribution
        ax = axes[idx, 1]
        real_returns = real_df['Close'].pct_change().dropna()
        synth_returns = synthetic_df['Close'].pct_change().dropna()
        sns.kdeplot(data=real_returns, label='Real', ax=ax)
        sns.kdeplot(data=synth_returns, label='Synthetic', ax=ax)
        ax.set_title(f'{ticker} - Return Distribution')
        ax.legend()
        
        # Volume plot
        ax = axes[idx, 2]
        ax.plot(real_df['Volume'].values[:100], label='Real', alpha=0.7)
        ax.plot(synthetic_df['Volume'].values[:100], label='Synthetic', alpha=0.7)
        ax.set_title(f'{ticker} - Volume Comparison')
        ax.legend()
    
    plt.tight_layout()
    return fig

In [4]:

# Example of how to use the code
# if __name__ == "__main__":
import sys
from pathlib import Path

import os

# Set the working directory to the root of the project
os.chdir(Path('..').resolve())  # Adjust this path as necessary

# Now append the src directory to the path
import sys
sys.path.append(str(Path('src').resolve()))

import numpy as np
import pandas as pd
# Now import the necessary modules
from config import Config
import argparse
import logging
from pathlib import Path
from config import Config, setup_logging
from model import get_models
from data_loader import (
    StockDataset,
    download_stock_data,
    get_sp500_tickers,
    prepare_data,
    create_dataloader
)
from trainer import GANTrainer
from evaluation import (
    generate_synthetic_data,
    evaluate_quality,
    plot_results,
    plot_training_history,
    analyze_distributions
)
# Initialize configuration and check the data directory
config = Config()
print(f"Data directory: {config.DATA_DIR}")

import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Fetch S&P 500 tickers and download stock data
tickers = get_sp500_tickers(config)
stock_data = download_stock_data(tickers, config)
# stock_data
# def robust_normalize(data: np.ndarray) -> np.ndarray:
#     """
#     Normalize the input data using the median and IQR method.

#     Args:
#         data (np.ndarray): The data to normalize, expected to be 2D.

#     Returns:
#         np.ndarray: Normalized data.
#     """
#     median = np.nanmedian(data, axis=0)  # Ignore NaNs
#     q75, q25 = np.nanpercentile(data, [75, 25], axis=0)
#     iqr = q75 - q25
#     normalized_data = (data - median) / (iqr + 1e-8)  # Avoid division by zero
#     return normalized_data

num_stocks = 3
rows_per_stock = 300

np.random.seed(31)

# Select stocks if num_stocks is specified
if num_stocks is not None:
    selected_stocks = np.random.choice(list(stock_data.keys()), num_stocks, replace=False)
else:
    selected_stocks = list(stock_data.keys())
print("Selected stocks:", selected_stocks)

# Create a mapping of stock names to unique IDs
stock_to_id = {stock: idx for idx, stock in enumerate(selected_stocks)}
print('stock_to_id:', stock_to_id)

# Get the full date range from the stock data
min_date = min(df.index.min() for df in stock_data.values())
max_date = max(df.index.max() for df in stock_data.values())
all_dates = pd.date_range(start=min_date, end=max_date, freq='B')

# Initialize dictionaries
data_dict = {}
aligned_data_dict = {}
mask_dict = {}
normalization_params = {}

for stock in selected_stocks:
    # Align stock data with the full date range
    df = stock_data[stock]
    df_aligned = df.reindex(all_dates)
    mask = df_aligned.notna().astype(int)
    
    data_dict[stock] = df_aligned.iloc[:rows_per_stock]
    # print('df_aligned')
    # display(df_aligned)
    
    # Normalize data
    # normalized_data = robust_normalize(df_aligned.values)
    normalized_df = pd.DataFrame(df_aligned, index=all_dates, columns=df.columns)
    # print('normalized_df')
    # display(normalized_df)
    normalized_df[mask == 0] = np.nan  # Keep NaNs where the original data is missing

    # Now, select only the specified number of non-NaN rows
    non_nan_rows = normalized_df.dropna().iloc[:rows_per_stock]

    # Store only the selected rows in the final dictionaries
    aligned_data_dict[stock] = non_nan_rows
    mask_dict[stock] = mask.loc[non_nan_rows.index]
    normalization_params[stock] = (
        np.nanmedian(df_aligned.values, axis=0),
        np.nanpercentile(df_aligned.values, 75, axis=0) - np.nanpercentile(df_aligned.values, 25, axis=0)
    )

gan = FinancialGAN(
    data_dict=data_dict,
    sequence_length=20,
    batch_size=32,
    noise_dim=100
)

# Train model
print("Training GAN...")
history = gan.train(epochs=100)

# Generate synthetic data
print("\nGenerating synthetic data...")
synthetic_data = gan.generate_synthetic_data(n_samples=100)

# Plot results
print("\nPlotting results...")
fig = plot_results(data_dict, synthetic_data)
plt.show()

# Plot training history
plt.figure(figsize=(10, 5))
plt.plot(history['d_loss'], label='Discriminator Loss')
plt.plot(history['g_loss'], label='Generator Loss')
plt.title('Training History')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

# Save model if needed
torch.save({
    'generator_state_dict': gan.generator.state_dict(),
    'discriminator_state_dict': gan.discriminator.state_dict(),
    'g_optimizer_state_dict': gan.g_optimizer.state_dict(),
    'd_optimizer_state_dict': gan.d_optimizer.state_dict(),
}, 'financial_gan_model.pth')

INFO:data_loader:Loaded data for 495 stocks


Data directory: data
data/stock_data.pkl
Selected stocks: ['QCOM' 'BRO' 'ACN']
stock_to_id: {'QCOM': 0, 'BRO': 1, 'ACN': 2}
Dataset initialized with shape: torch.Size([3, 300, 5])
Number of stocks: 3
Initializing GAN with 3 stocks


/Users/lucijagregov/Documents/gan_finance/.venv_finance/lib/python3.11/site-packages/torch/nn/modules/transformer.py:286: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


Training GAN...


  0%|          | 0/100 [00:00<?, ?it/s]

...

  1%|          | 1/100 [02:01<3:19:53, 121.15s/it]

NaN detected at epoch 0, batch 7
Epoch [0/100] D_loss: nan G_loss: nan
Initial real_data: torch.Size([32, 3, 20, 5])
After noise projection: torch.Size([32, 256])
After adding time dimension: torch.Size([32, 20, 256])
After time embedding: torch.Size([32, 20, 256])
After stock embedding: torch.Size([32, 3, 20, 256])
Before transformer: torch.Size([20, 96, 256])
After transformer: torch.Size([20, 96, 256])
After reshaping back: torch.Size([32, 3, 20, 256])
Generated prices: torch.Size([32, 3, 20, 4])
Generated volume: torch.Size([32, 3, 20, 1])
Final output: torch.Size([32, 3, 20, 5])
Generated fake_data: torch.Size([32, 3, 20, 5])
Discriminator input: torch.Size([32, 3, 20, 5])
After feature extraction: torch.Size([32, 3, 20, 256])
After time embedding: torch.Size([32, 3, 20, 256])
After stock embedding: torch.Size([32, 3, 20, 256])
Before transformer: torch.Size([20, 96, 256])
After transformer: torch.Size([20, 96, 256])
After pooling: torch.Size([96, 256])
Final output: torch.Size([9

  1%|          | 1/100 [02:22<3:54:32, 142.15s/it]

Discriminator input: torch.Size([32, 3, 20, 5])
After feature extraction: torch.Size([32, 3, 20, 256])
After time embedding: torch.Size([32, 3, 20, 256])
After stock embedding: torch.Size([32, 3, 20, 256])
Before transformer: torch.Size([20, 96, 256])


KeyboardInterrupt: 

In [5]:
data_dict

{'QCOM':                  Open       High        Low      Close      Volume
 2020-01-02  89.050003  89.809998  88.080002  88.690002   8413900.0
 2020-01-03  87.260002  87.639999  86.440002  87.019997   8340300.0
 2020-01-06  85.910004  86.550003  85.540001  86.510002   8381400.0
 2020-01-07  87.040001  89.489998  86.910004  88.970001   8377400.0
 2020-01-08  88.900002  89.470001  87.919998  88.709999   7619900.0
 ...               ...        ...        ...        ...         ...
 2020-05-14  77.500000  80.000000  76.470001  79.870003   9956800.0
 2020-05-15  74.669998  77.699997  74.370003  75.769997  29597800.0
 2020-05-18  77.269997  80.180000  77.209999  79.940002  13191000.0
 2020-05-19  79.750000  80.150002  78.000000  78.089996   8118400.0
 2020-05-20  79.959999  81.959999  79.690002  80.629997  11379200.0
 
 [100 rows x 5 columns],
 'BRO':                  Open       High        Low      Close     Volume
 2020-01-02  39.619999  39.669998  39.200001  39.610001  1354300.0
 2020-01